In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.neighbors import KernelDensity
import matplotlib.pyplot as plt

In [11]:
side_lengths = torch.linspace(0, 50, 100)

models = []
for side_length in side_lengths:
    model = [
        [side_length, -side_length, side_length, -side_length],
        [side_length, side_length, -side_length, -side_length],
        [0.0, 0.0, 0.0, 0.0],
    ]
    models.append(model)
models = torch.tensor(models)

In [12]:
models.shape

torch.Size([100, 3, 4])

In [13]:
models_selected = models

models_transposed = models_selected.transpose(1, 2)

models_transposed.shape


torch.Size([100, 4, 3])

In [ ]:
class DistanceGNN(nn.Module):
    """A simple invariant GNN using pairwise distances only."""
    def __init__(self, in_dim=1, hidden_dim=64, out_dim=8):
        'in_dim = number of features per node'
        'hidden dimension is the size of intermediate representations'
        'graph embedding size for flow input'
        'x: [num_nodes, in_dim] → input node features
        'edge_attr: [num_edges, 1] → distance features'
        'node_embeddings: [num_nodes, out_dim] → node-level'
        'graph_embedding: [out_dim] → input to density estimator'

        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, coords):
        """
        coords: [N, 3] tensor for one protein conformation.
        """
        N = coords.shape[0]
        print(N)
        dists = torch.cdist(coords, coords)  # [N, N]
        print(dists.shape)
        mask = ~torch.eye(N, dtype=torch.bool)  # remove self-loops
        print(mask.shape)
        d = dists[mask].unsqueeze(-1)           # [N*(N-1), 1] edge_attr, edge features, that we aggregate, pairwise distances 
        print(d.shape)

        # Compute pairwise "messages" from distances
        edge_features = self.edge_mlp(d)        # [E, hidden_dim]
        print(edge_features.shape)
        edge_features = edge_features.view(N, N - 1, -1).mean(dim=1)  # mean over neighbors
        print(edge_features.shape)

        # Aggregate to node embeddings, then global mean pool
        node_embeddings = self.node_mlp(edge_features)
        print(node_embeddings.shape)
        graph_embedding = node_embeddings.mean(dim=0)
        print(graph_embedding.shape)
        return graph_embedding

In [21]:
model = DistanceGNN()
model.eval()
with torch.no_grad():
    z = model(coords)
#embeddings = []
#for coords in frames:
#    with torch.no_grad():
#        z = model(coords)
#    embeddings.append(z.numpy())#

#embeddings = np.vstack(embeddings)
#print("Embeddings shape:", embeddings.shape)

4
torch.Size([4, 4])
torch.Size([4, 4])
torch.Size([12, 1])
torch.Size([12, 64])
torch.Size([4, 64])
torch.Size([4, 8])
torch.Size([8])


In [17]:
z.shape

torch.Size([8])

In [68]:
def coords_to_graph(coords, cutoff=5):
    N = coords.shape[0]
    x = torch.ones((N, 1)) #Node Features [num_nodes, node_feature_dim], features for each node in the graph. (Residue type later)
    dist = torch.cdist(coords, coords)
    #edge_mask = ~torch.eye(N, dtype=torch.bool)
    edge_mask = (dist < cutoff) & (~torch.eye(N, dtype=torch.bool)) #True only for pairs that are below the cutoff AND are not self-loops
    edge_index = edge_mask.nonzero(as_tuple=False).t()        # [2, num_edges] which nodes are connected by edges, based on distance cutoff 
    edge_attr = dist[edge_index[0], edge_index[1]].unsqueeze(-1) # Edge Features, pairwise distances between nodes [num_edges, 1]
    return x, edge_index, edge_attr




In [69]:
x, edge_index, edge_attr = coords_to_graph(coords)

In [70]:
edge_attr

tensor([[3.0303],
        [3.0303],
        [4.2855],
        [3.0303],
        [4.2855],
        [3.0303],
        [3.0303],
        [4.2855],
        [3.0303],
        [4.2855],
        [3.0303],
        [3.0303]])

In [71]:
class VectorizedDistanceGNN(nn.Module):
    def __init__(self, in_dim=1, hidden_dim=64, out_dim=16):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )
    
    def forward(self, x, edge_index, edge_attr):
        N = x.shape[0]
        num_edges = edge_index.shape[1]

        # -----------------------------
        # Edge messages
        # -----------------------------
        # edge_attr shape [num_edges, edge_feature_dim], edge_feature_dim = 1 pairwise distances 
        messages = self.edge_mlp(edge_attr)  # [num_edges, hidden_dim], one edge and contains its features
        # edge features into representation
        # -----------------------------
        # Aggregate messages per target node (vectorized)
        # -----------------------------
        target_nodes = edge_index[1]                    # [num_edges]
        node_messages = torch.zeros((N, messages.shape[1]), device=x.device) #tensor to accumulate incoming messages for each node.
        counts = torch.zeros(N, device=x.device)

        # Use scatter_add for fast accumulation
        node_messages = node_messages.index_add(0, target_nodes, messages) 
        counts = counts.index_add(0, target_nodes, torch.ones(num_edges, device=x.device))
        counts = counts.clamp(min=1.0).unsqueeze(-1)
        node_features = node_messages / counts          # mean aggregation

        # -----------------------------
        # Node MLP
        # -----------------------------
        node_embeddings = self.node_mlp(node_features)  # [N, out_dim]

        # -----------------------------
        # Graph-level embedding
        # -----------------------------
        graph_embedding = node_embeddings.mean(dim=0)  # [out_dim]
        return graph_embedding




In [72]:

model = VectorizedDistanceGNN()
model.eval()


x, edge_index, edge_attr = coords_to_graph(coords)
with torch.no_grad():
    z = model(x, edge_index, edge_attr)


#embeddings = np.vstack(embeddings)
#print("Graph embeddings shape:", embeddings.shape)


In [26]:
import torch
import torch.nn as nn

class FullyVectorizedDistanceGNN(nn.Module):
    def __init__(self, in_dim=1, hidden_dim=64, out_dim=16):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    @staticmethod
    def batch_coords_to_graph(coords_batch, cutoff=5):
        """
        Fully vectorized graph construction.
        coords_batch: [B, N, 3]
        Returns:
            x: [B*N,1] node features
            edge_index: [2, total_edges]
            edge_attr: [total_edges,1]
            batch: [B*N] node-to-graph mapping
        """
        B, N, _ = coords_batch.shape
        
        device = coords_batch.device

        # Node features
        x = torch.ones(B * N, 1, device=device)
        batch = torch.arange(B, device=device).repeat_interleave(N)

        # Compute all pairwise distances per graph using broadcasting
        coords_i = coords_batch.unsqueeze(2)  # [B,N,1,3]
        coords_j = coords_batch.unsqueeze(1)  # [B,1,N,3]
        dist = torch.norm(coords_i - coords_j, dim=-1)  # [B,N,N]

        # Mask: no self-loops and within cutoff
        mask = (dist < cutoff) & (~torch.eye(N, dtype=torch.bool, device=device).unsqueeze(0))
        # Get edge indices and attributes
        b_idx, i_idx, j_idx = mask.nonzero(as_tuple=True)  # b, src, tgt
        edge_index = torch.stack([i_idx + b_idx * N, j_idx + b_idx * N], dim=0)  # [2, total_edges]
        edge_attr = dist[b_idx, i_idx, j_idx].unsqueeze(-1)  # [total_edges,1]

        return x, edge_index, edge_attr, batch

    def forward(self, coords_batch, cutoff=5):
        B, N, _ = coords_batch.shape
        print(B, N, _ )
        device = coords_batch.device

        # Build graph
        x, edge_index, edge_attr, batch = self.batch_coords_to_graph(coords_batch, cutoff)
        print('x, edge_index, edge_attr, batch',x.shape, edge_index.shape, edge_attr.shape, batch.shape)


        # Edge MLP
        messages = self.edge_mlp(edge_attr)  # [total_edges, hidden_dim]
        print(messages.shape)

        # Aggregate messages per node
        target_nodes = edge_index[1]
        node_messages = torch.zeros((B*N, messages.shape[1]), device=device)
        counts = torch.zeros(B*N, device=device)
        node_messages = node_messages.index_add(0, target_nodes, messages)
        counts = counts.index_add(0, target_nodes, torch.ones_like(target_nodes, dtype=torch.float))
        counts = counts.clamp(min=1.0).unsqueeze(-1)
        node_features = node_messages / counts

        # Node MLP
        node_embeddings = self.node_mlp(node_features)  # [B*N, out_dim]

        # Graph-level mean pooling
        graph_embeddings = torch.zeros((B, node_embeddings.shape[1]), device=device)
        graph_embeddings = graph_embeddings.index_add(0, batch, node_embeddings)
        counts_graph = torch.zeros(B, device=device)
        counts_graph = counts_graph.index_add(0, batch, torch.ones(B*N, device=device))
        counts_graph = counts_graph.clamp(min=1.0).unsqueeze(-1)
        graph_embeddings = graph_embeddings / counts_graph

        return graph_embeddings


In [27]:
nn = FullyVectorizedDistanceGNN()
nn.eval()

FullyVectorizedDistanceGNN(
  (edge_mlp): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
  (node_mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
  )
)

In [28]:
nn.forward(models_transposed)

100 4 3
x, edge_index, edge_attr, batch torch.Size([400, 1]) torch.Size([2, 56]) torch.Size([56, 1]) torch.Size([400])
torch.Size([56, 64])


tensor([[ 0.0603, -0.0574,  0.0525,  ...,  0.0280,  0.0797, -0.0838],
        [-0.0230, -0.1270,  0.1037,  ...,  0.0513,  0.0990, -0.1443],
        [-0.0869, -0.1253,  0.1547,  ...,  0.1233,  0.1461, -0.2025],
        ...,
        [ 0.0999, -0.0844,  0.0235,  ..., -0.0653,  0.0802, -0.0949],
        [ 0.0999, -0.0844,  0.0235,  ..., -0.0653,  0.0802, -0.0949],
        [ 0.0999, -0.0844,  0.0235,  ..., -0.0653,  0.0802, -0.0949]],
       grad_fn=<DivBackward0>)

In [ ]:
models_transposed